<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">


# The Data Scientist
## Book 4 · Software Engineering, Reproducibility, and Deployment Basics
### Notebook 04 · Portfolio Capstone

© Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://thedatascientist.dev | https://linktr.ee/dyjh


### Why this notebook exists
The capstone is meant to look like a small professional project, not a scattered collection of experiments.

This cell loads the shipped project-health data, validates the columns, and prints a compact summary before the next step.


This cell prepares the notebook for local or Colab execution. It finds the book root, clones the companion repo when needed, and makes the working directory predictable.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

BOOK_NAME = "4_data"  # book folder to locate locally or after clone
REPO_URL = "https://github.com/yhilpisch/tdscode"  # companion repo with notebooks, data, and code

cwd = Path.cwd().resolve()  # start from the current working directory
BOOK_ROOT = None  # filled once a matching book directory is found
for candidate in [cwd, *cwd.parents]:  # search upward for the book root
    if candidate.name == BOOK_NAME and (candidate / "notebooks").exists():  # local book root
        BOOK_ROOT = candidate
        break
    if (candidate / BOOK_NAME / "notebooks").exists():  # repo/book layout
        BOOK_ROOT = candidate / BOOK_NAME
        break

if BOOK_ROOT is None:
    repo_root = Path("/content/tdscode")  # Colab clone location
    if not repo_root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_root)], check=True)  # fetch the companion repo once
    BOOK_ROOT = repo_root / BOOK_NAME  # book folder inside the clone

os.chdir(BOOK_ROOT)  # keep relative paths anchored on the book root
if str(BOOK_ROOT) not in sys.path:  # allow src/ imports
    sys.path.insert(0, str(BOOK_ROOT))  # allow src/ imports

code_dir = BOOK_ROOT / "code"  # helper scripts live here
if code_dir.exists() and str(code_dir) not in sys.path:  # make helper scripts importable
    sys.path.insert(0, str(code_dir))

requirements = BOOK_ROOT / "requirements.txt"  # install only when present
if requirements.exists() and "google.colab" in sys.modules:  # keep Colab in sync with the book
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)  # keep Colab in sync with the book

print("Book root:", BOOK_ROOT)


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()  # start from the current working directory
BOOK_ROOT = cwd
for candidate in [cwd, cwd.parent, cwd / "4_data"]:  # probe the likely launch locations
    if (candidate / "src").exists() or (candidate / "code").exists():
        BOOK_ROOT = candidate
        break
if str(BOOK_ROOT) not in sys.path:  # make the book root importable
    sys.path.insert(0, str(BOOK_ROOT))

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.data_checks import load_project_health, project_health_summary, validate_project_health  # reuse the shared validation helpers


This cell loads the shipped project-health data, validates the columns, and prints a compact summary before the next step.


In [ ]:
frame = load_project_health()
validate_project_health(frame)
print("Summary:", project_health_summary(frame))
print()
print(frame.head().to_string(index=False))


This cell builds the supervised-learning baseline by selecting features, keeping the target separate, and preparing a train/test split.


In [ ]:
feature_columns = ["domain", "owner_type", "active_weeks", "notebooks", "scripts", "test_files", "dashboard_views", "stale_days", "issue_count", "has_readme", "has_requirements"]  # define the model inputs
target = "delivery_risk"

X = frame[feature_columns]
y = frame[target]

numeric = ["active_weeks", "notebooks", "scripts", "test_files", "dashboard_views", "stale_days", "issue_count"]  # list the numeric features
categorical = ["domain", "owner_type", "has_readme", "has_requirements"]  # list the categorical features

preprocessor = ColumnTransformer([  # scale numeric features and one-hot encode categoricals
    ("num", StandardScaler(), numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
])
model = Pipeline([  # keep preprocessing and classifier together
    ("pre", preprocessor),
    ("clf", LogisticRegression(max_iter=1000)),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)  # split off held-out data
model.fit(X_train, y_train)  # fit on the training split
preds = model.predict(X_test)  # score the held-out rows

print("accuracy =", round(accuracy_score(y_test, preds), 3))
print()
print(classification_report(y_test, preds, digits=3))


This cell saves a short project report so the notebook ends with a concrete file on disk.


In [ ]:
report_path = BOOK_ROOT / "artifacts" / "capstone_summary.md"  # create the report file path
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("# Module 4 Capstone\n\nThis project packages a small project-health dataset, reusable checks, a Streamlit dashboard, and a short evaluation notebook.\n")  # write the portfolio note
print(report_path.resolve())


### Report Summary
The final project should explain what problem it solves, how to run it, what the dashboard adds, and what the next improvement would be.

### Where We Are Heading Next
The module ends here. The next real step is to publish the notebook, the code, the dataset, and the README as one coherent portfolio repository.

<img src="https://hilpisch.com/tds_logo.png" width="20%" align="right">
